<a href="https://colab.research.google.com/github/annaannaR/NOTEBOOKS-FROM-SCHOOL/blob/main/AMS/Machine%20Learning/PART_3/MLP3_3_Neural_Networks_github.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Training and testing Neural Network - using MNIST dataset

The MNIST (Modified National Institute of Standards and Technology) dataset is a dataset of 60,000 small square 28×28 pixel grayscale images of handwritten single digits between 0 and 9.

The MNIST handwritten digit classification problem is a standard dataset often used when starting out with computer vision and deep learning.

Our task is to classify an image of a digit into one of 10 classes representing values from 0-9.

Let's first import the necessary methods.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from tensorflow import keras
from keras import layers, models, datasets

We can now load the dataset, which is already devided into a training- and test set.

In [ ]:
# Load the data and split it between train and test sets
(x_train, y_train), (x_test, y_test) = datasets.mnist.load_data()

Let's see what our dataset looks like to better understand what problem we are trying to solve.

In [ ]:
print('Train: X=%s, y=%s' % (x_train.shape, y_train.shape))
print('Test: X=%s, y=%s' % (x_test.shape, y_test.shape))

We see we have a trainingset of 60000 images, and 10000 images to test on.

The (28, 28) dimension refers to the number of pixels per image, 28 rows of 28 points, so for each image we have 28x28 = 784 datapoints, saved in a NumPy-array.
The value of the datapoint represents their grayscale.

We can almost recognise the numbers just looking at the datapoints  (click 'Show data'). Although the numbers are distorted because Colab only outputs 13 values in one row, instead of the 28 we need now to get a square-image:

In [ ]:
x_train[1]

Let's just look at the images instead:

In [ ]:
# plot first few images
for i in range(9):
  # Set the grid
	plt.subplot(331 + i)
	# plot raw pixel data
	plt.imshow(x_train[i], cmap=plt.get_cmap('gray'))
# show the figure
plt.show()

So we have data representing handwritten digits in x_train/x_test (the feature datasets). The question is: can we train a neural network to correctly identify these handwritten numbers?

The correct answers we use for training and testing our network are of course stored in the y_train and y_test datasets:

In [ ]:
for i in range(9):
	print("The correct class is:", y_train[i])

#Prepare the data

We are going to scale the data to be in a range from 0 to 1, instead of the grayscale it is now (i.a. for computational reasons).

And when working with images the datapoints should be shaped like a matrix: color X width X heigth (see explanation [here](https://stackoverflow.com/questions/65202011/why-are-the-mnist-images-1x28x28-tensors)).
Although for black-and-white images it doesn't change anything, Keras expects 3-dimensions, so we add one.

In [ ]:
# Scale images to the [0, 1] range
x_train = x_train.astype("float32") / 255
x_test = x_test.astype("float32") / 255

# Make sure images have shape (28, 28, 1)
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)
print("x_train shape:", x_train.shape)

Although we are working with numbers, we still want the model to treat this problem like a classification-problem. We want to classify the handwritten digits being a '0', a '1', a '2', etc.

To do this we will convert our classes (10 options, numbers 0-9) to a binary array (similar to what we did with OneHotEncoding). So all answers are converted to an array of length 10, with values being 0 or 1 depending on which number it represents:

In [ ]:
print("The correct digit is:", y_train[1])
num_classes = 10

# convert class vectors to binary class matrices
y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)

# look at result
print("Now converted to a binary array:", y_train[1])

##Designing the Neural Network
Now we can move on to the modeling part. Just like with the other Machine Learning models we first need to initiate a model, and after that we can train and test it.

With SciKit-learn this was the part where we took the build-in algorithm and named it, like:

`my_nn_model = NeuralNetwork()`

However, initialising a Neural Network with Keras is a bit more work. We need to give it a lot more information upfront, to define what the network should look like.

For example, we need to define:


*   the number of hidden layers
*   the number of neurons per layer
*   which activation function(s)




This 'designing' of your neural network is the hardest part. There are many, almost endless, options. And how do you know upfront what will work well?
Basically, you don't. So you just start with some standard options, that usually perform well on other problems, and go from there.


You will train and test your model, and go back again to designing it, adding/removing layers, changing the number of neurons, the learning rate, the loss-function, the number of epochs etc.
Unfortunately this goes beyond the scope of this training, we can only cover an introduction into Neural Networks, but this article should be a really good guide when creating your own NNs: https://towardsdatascience.com/designing-your-neural-networks-a5e4617027ed.


Let's start by defining the type of network, and setting the Input-layer:

In [ ]:
# Initialize as a sequential model (going from one layer to the other)
nn_model = keras.Sequential()

# Set the Input-layer, in which we flatten the input shape from being a matrix of 28x28, to 784 input-neurons
nn_model.add(layers.Input(shape=(28, 28)))
nn_model.add(layers.Flatten())

We will train a model with 2 hidden layers. The first one having 32 neurons, the second one 64 neurons.
We use the Sigmoid activation function in these layers.
We want each neuron to receive input from all the neurons of the previous layer, this is called a 'Dense' layer.

In [ ]:
# Add the hidden-layers
nn_model.add(layers.Dense(32, activation='sigmoid'))
nn_model.add(layers.Dense(64, activation='sigmoid'))

And finally we add the output layer. This layer needs 10 neurons, corresponding to the number of classes we have. The model will output a probability to all these 10 neurons: the probability that the particular digit is this class.

We use a softmax activation function to make sure the probabilities sum to 1 (with a perfect classification of course being a 1 on the correct class-neuron and 0 on the others).

In [ ]:
# Add the output layer
nn_model.add(layers.Dense(10, activation='softmax'))

We can look at the details of our designed model by using .summary().
We see all the layers we added, the number of neurons in those layers, and the number of parameters (weights) that get trained correspondingly.

In [ ]:
nn_model.summary()

So already with this fairly simple network (only 2 hidden layers) we are training almost 30000 parameters! You see why this get's called deep learning.

NB: Try to think of the reason why the number of parameters isn't just the number of neurons of the former layer X the number of neurons in the current layer.

After designing the Neural Network, we need to compile it with the following code. Setting the loss-function (which function to use to calculate the error), which optimizer to use in the Back Propagation, and how to score the model:

In [ ]:
nn_model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

##Training
Now we can train our model, just like we did with SciKit-learn, using a .fit() function.

We will do 5 epochs, so optimizing our weights (using all the training data) 5 times.

By already passing the test-data into the arguments, the model will test itself on the test-data.

In [ ]:
epochs = 5

nn_model.fit(x_train, y_train, epochs=epochs, validation_data=(x_test,y_test))

NB: For computational reasons not all 60000 datapoints can be handled at the same time. The data gets divided into smaller batches and the models iterates over these batches.
So for our model each epoch has 1875 iterations, because the default batch-size is 32 datapoints (60000/32=1875).

Our model accuracy is really good, around 96 percent of the digits get classified correctly!

And we see the difference between the training accuracy and the test accuracy is small, so our model doesn't seem to be over-/underfitting.

In [ ]:
# Take a look at the actual predictions
predictions = nn_model.predict(x_test)
print(predictions)

As you can see in the code above, we can again just use the function .predict() to get the actual predictions.

But remember these are the probabilities that the digit belongs to a certain class (0-9). So for each digit you get 10 probabilities.
We can convert these results to actually predicting one number per digit by using the NumPy function '[argmax](https://numpy.org/doc/stable/reference/generated/numpy.argmax.html)'. The result is the number with the highest probability.

In [ ]:
predictions = np.argmax(predictions, axis=1)
print(predictions)

We visualize the predictions by plotting the images, and using the predictions as title:

In [ ]:
x_test = np.squeeze(x_test)

In [ ]:
fig, axes = plt.subplots(ncols=10, sharex=False,
                         sharey=True, figsize=(20, 4))
for i in range(10):
    # Show our predictions
    axes[i].set_title(predictions[i])

    # Show the original images
    axes[i].imshow(x_test[i], cmap='gray')
    axes[i].get_xaxis().set_visible(False)
    axes[i].get_yaxis().set_visible(False)
plt.show()

Looks like these first 10 predictions are all correct! (which was to be expected with such a high accuracy)

So this was your first introduction into designing, training and testing Neural Networks. Feel free to play around with the Neural Network: adding layers, using other activation functions, other optimizing algoritms, batch_sizes, epochs etc.

(only once you've copied this notebook to your own Drive of course)